# Cuaderno 06. Orden de lectura guiado por el sentido: bloques, OCR y un modelo de lenguaje

Reconstrucción del método de [Vidal-Gorène, Tomeh y Khurshudyan (2026)](https://arxiv.org/html/2607.00596v1) con modelos genéricos.

## 0. Elegir OCR y modelo de lenguaje, e instalar

| Variable | Valores | Diferencia |
|---|---|---|
| `OCR` | `"tesseract"` (por defecto) | OCR clásico; se instala en segundos |
| | `"paddle"` | PP-OCRv5; la instalación tarda unos 9 minutos |
| `LLM` | `"ollama"` (por defecto) | Ministral-3 8B dentro del entorno: **el texto no sale de Colab** |
| | `"gemini"` | Gemini 3.5 Flash-Lite por API. Requiere una clave y **el texto se envía a Google** |

Para usar Gemini, crear una clave en [Google AI Studio](https://aistudio.google.com/apikey) y guardarla en Colab como secreto con el nombre `GEMINI_API_KEY` (ícono de la llave, en la barra lateral). Fuera de Colab, definir la variable de entorno `GEMINI_API_KEY`.

In [ ]:
OCR = "tesseract"
LLM = "ollama"

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run("apt-get -qq install -y tesseract-ocr tesseract-ocr-spa > /dev/null", shell=True, check=True)
    paquetes = ["doclayout-yolo==0.0.4", "pypdfium2", "pytesseract"]
    if LLM == "gemini":
        paquetes.append("google-genai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *paquetes], check=True)
    if OCR == "paddle":
        import shutil

        GPU = shutil.which("nvidia-smi") is not None
        if GPU:
            paddle = ["paddlepaddle-gpu==3.2.1", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/"]
        else:
            paddle = ["paddlepaddle==3.2.1"]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *paddle], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "paddleocr==3.7.0"], check=True)
        # PaddlePaddle-GPU instala la biblioteca NCCL de CUDA 12 sobre la de CUDA 13 que usa el PyTorch de Colab:
        # se restituye la versión que pide PyTorch (compatible con Paddle). Ver el cuaderno 02.
        import importlib.metadata as metadata

        nccl = [r.split(";")[0].replace(" ", "") for r in (metadata.requires("torch") or []) if r.startswith("nvidia-nccl")]
        if nccl:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", *nccl], check=True)
    if LLM == "ollama":
        subprocess.run("apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    print("Instalado: doclayout-yolo, Tesseract", "+ PaddleOCR" if OCR == "paddle" else "", "+ Ollama" if LLM == "ollama" else "+ google-genai")
else:
    print("Fuera de Colab no se instala nada: preparar el entorno según cuadernos/README.md.")

## 1. Elegir las imágenes

`SELECCION` define qué se procesa:

| Valor | Qué procesa |
|---|---|
| `"comun"` | la página común (*Centro Gallego*, Montevideo, 1917) |
| `"prensa"` | todas las páginas de prensa del taller |
| `"manuscritos"` | cartas, actas y otros manuscritos del taller |
| `"todas"` | prensa y manuscritos |
| `"propias"` | tus imágenes o PDF; en Colab aparecen dos botones para subir **varios archivos** o **una carpeta completa**; fuera de Colab se leen de `mis_imagenes/` (incluidas sus subcarpetas) |

Cada página de un PDF se convierte en una imagen PNG (`DPI_PDF = 200`).

In [ ]:
SELECCION = "comun"

In [ ]:
import csv
import sys
import urllib.request
import uuid
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from PIL import Image

EN_COLAB = "google.colab" in sys.modules
REPO_RAW = "https://raw.githubusercontent.com/agusnieto77/sicss-uruguay-2026/main/cuadernos/imagenes/"
BASE = Path("/content/taller") if EN_COLAB else Path.cwd()
CARPETA_IMAGENES = BASE / "imagenes"
EXTENSIONES_IMAGEN = {".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff"}
EXTENSIONES = EXTENSIONES_IMAGEN | {".pdf"}
DPI_PDF = 200  # resolución con la que cada página de un PDF se convierte en imagen


def traer(relativa):
    """Ruta local de un archivo del taller; en Colab lo descarga del repositorio la primera vez."""
    destino = CARPETA_IMAGENES / relativa
    if EN_COLAB and not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + relativa, destino)
    if not destino.exists():
        raise FileNotFoundError(f"No encuentro {destino}. Abrí el cuaderno desde la carpeta cuadernos/ del repositorio.")
    return destino


def mostrar(imagen, ancho=1000):
    """Muestra una imagen PIL reducida a un ancho cómodo para la pantalla."""
    if imagen.width > ancho:
        imagen = imagen.resize((ancho, round(imagen.height * ancho / imagen.width)))
    display(imagen)


def admitido(nombre):
    """Imágenes y PDF, sin archivos ocultos (por ejemplo, los `._foto.jpg` que deja macOS en las carpetas)."""
    return not Path(nombre).name.startswith(".") and Path(nombre).suffix.lower() in EXTENSIONES


def pdf_a_imagenes(pdf, carpeta):
    """Convierte cada página de un PDF en una imagen PNG y devuelve sus rutas."""
    import pypdfium2 as pdfium

    carpeta.mkdir(parents=True, exist_ok=True)
    documento = pdfium.PdfDocument(pdf)
    paginas = []
    for indice in range(len(documento)):
        pagina = documento[indice]
        destino = carpeta / f"{pdf.stem}_p{indice + 1:03d}.png"
        pagina.render(scale=DPI_PDF / 72).to_pil().convert("RGB").save(destino)
        pagina.close()
        paginas.append(destino)
    documento.close()
    return paginas


SELECTOR_COLAB = """
<div style="display:flex; gap:2em; align-items:center; flex-wrap:wrap">
  <label><b>Elegir archivos</b> (imágenes y PDF):
    <input type="file" id="__ID__-archivos" multiple accept="image/*,.tif,.tiff,.pdf"></label>
  <span><b>o una carpeta completa</b>:
    <input type="file" id="__ID__-carpeta" webkitdirectory style="display:none"
      onchange="document.getElementById('__ID__-estado').textContent = this.files.length
        ? this.files[0].webkitRelativePath.split('/')[0] + ' (' + this.files.length + ' archivos)' : ''">
    <button type="button" onclick="document.getElementById('__ID__-carpeta').click()">Seleccionar carpeta</button>
    <span id="__ID__-estado">Ninguna carpeta seleccionada</span></span>
</div>
<script>
window.subidaTaller = {
  elegir: (id) => new Promise((resolve) => {
    for (const sufijo of ["archivos", "carpeta"]) {
      document.getElementById(id + "-" + sufijo).addEventListener("change", (evento) => {
        window.subidaTaller.lista = Array.from(evento.target.files);
        resolve(window.subidaTaller.lista.map((f) => f.webkitRelativePath || f.name));
      });
    }
  }),
  trozo: (indice, inicio, fin) => new Promise((resolve, reject) => {
    const lector = new FileReader();
    lector.onload = () => resolve(lector.result.split(",")[1] || "");
    lector.onerror = () => reject(lector.error);
    lector.readAsDataURL(window.subidaTaller.lista[indice].slice(inicio, fin));
  }),
  tamano: (indice) => window.subidaTaller.lista[indice].size,
};
</script>
"""


def subir_en_colab(carpeta, trozo=1024 * 1024):
    """Diálogo de Colab para elegir varios archivos o una carpeta; guarda imágenes y PDF en `carpeta`."""
    import base64

    from google.colab import output

    identificador = "subida-" + uuid.uuid4().hex
    display(HTML(SELECTOR_COLAB.replace("__ID__", identificador)))
    nombres = output.eval_js(f'window.subidaTaller.elegir("{identificador}")')
    for indice, nombre in enumerate(nombres):
        if not admitido(nombre):
            print("Ignorado (no es imagen ni PDF):", nombre)
            continue
        destino = carpeta.joinpath(*Path(nombre).parts)  # una carpeta conserva sus subcarpetas
        destino.parent.mkdir(parents=True, exist_ok=True)
        tamano = output.eval_js(f"window.subidaTaller.tamano({indice})")
        with open(destino, "wb") as archivo:
            for inicio in range(0, tamano, trozo):
                datos = output.eval_js(f"window.subidaTaller.trozo({indice}, {inicio}, {inicio + trozo})")
                archivo.write(base64.b64decode(datos))
        print(f"Subido: {nombre} ({tamano / 1e6:.1f} MB)")


FICHAS = {}
if SELECCION == "propias":
    carpeta_propia = BASE / "mis_imagenes"
    carpeta_propia.mkdir(parents=True, exist_ok=True)
    if EN_COLAB:
        subir_en_colab(carpeta_propia)
    IMAGENES = []
    for archivo in sorted(p for p in carpeta_propia.rglob("*") if p.is_file() and admitido(p.name)):
        if archivo.suffix.lower() == ".pdf":
            paginas = pdf_a_imagenes(archivo, BASE / "paginas_pdf")
            print(f"{archivo.name}: {len(paginas)} página(s) convertidas a imagen a {DPI_PDF} dpi")
            IMAGENES += paginas
        else:
            IMAGENES.append(archivo)
else:
    with open(traer("fuentes.csv"), encoding="utf-8-sig") as archivo:
        fichas = list(csv.DictReader(archivo, delimiter=";"))
    filtros = {
        "comun": lambda f: f["pagina_comun"] == "si",
        "prensa": lambda f: f["tipo"] == "prensa",
        "manuscritos": lambda f: f["tipo"] != "prensa",
        "todas": lambda f: True,
    }
    if SELECCION not in filtros:
        raise ValueError(f"SELECCION debe ser uno de {sorted(filtros) + ['propias']}")
    elegidas = [f for f in fichas if filtros[SELECCION](f)]
    IMAGENES = [traer(f["archivo"]) for f in elegidas]
    FICHAS = {ruta.name: ficha for ruta, ficha in zip(IMAGENES, elegidas)}
    display(pd.DataFrame(elegidas)[["archivo", "tipo", "titulo", "fecha", "lugar", "institucion", "licencia"]])

assert IMAGENES, "No hay imágenes para procesar: revisá SELECCION o la carpeta mis_imagenes/."
print(f"{len(IMAGENES)} imagen(es):", [p.name for p in IMAGENES])

## 2. Preparar el modelo de lenguaje

Con `LLM = "ollama"` se inicia un servidor local de Ollama y se descarga `ministral-3:8b` (unos 6 GB). Con `LLM = "gemini"` se usa la clave guardada como secreto. `preguntar()` envía una consulta y devuelve la respuesta del modelo, con temperatura 0.

In [ ]:
import os
import time

import requests

MODELO_OLLAMA = "ministral-3:8b"
# Gemini 3 Flash, el del artículo, está deprecado. 3.5 Flash-Lite es el modelo vigente más barato;
# "gemini-3.8-flash" es el Flash más reciente, más parecido al del artículo y más caro.
MODELO_GEMINI = "gemini-3.5-flash-lite"
OLLAMA = "http://localhost:11434"

if LLM == "ollama":
    def ollama_activo():
        try:
            return requests.get(f"{OLLAMA}/api/version", timeout=3).ok
        except requests.RequestException:
            return False

    if not ollama_activo():
        subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
        for _ in range(60):
            if ollama_activo():
                break
            time.sleep(2)
        else:
            raise RuntimeError("Ollama no respondió; revisar ollama.log")
    subprocess.run(["ollama", "pull", MODELO_OLLAMA], check=True, stdout=subprocess.DEVNULL)
    requests.post(f"{OLLAMA}/api/generate", json={"model": MODELO_OLLAMA, "keep_alive": "60m"}, timeout=600).raise_for_status()
    VERSION_LLM = f"Ollama {requests.get(f'{OLLAMA}/api/version').json()['version']} · {MODELO_OLLAMA}"

    def preguntar(consulta):
        respuesta = requests.post(f"{OLLAMA}/api/generate", timeout=300, json={
            "model": MODELO_OLLAMA, "prompt": consulta, "stream": False, "options": {"temperature": 0, "num_predict": 8},
        })
        respuesta.raise_for_status()
        return respuesta.json()["response"]

elif LLM == "gemini":
    import importlib.metadata as metadata

    from google import genai
    from google.genai import types

    if EN_COLAB:
        from google.colab import userdata

        CLAVE = userdata.get("GEMINI_API_KEY")
    else:
        CLAVE = os.environ["GEMINI_API_KEY"]
    cliente = genai.Client(api_key=CLAVE)
    VERSION_LLM = f"google-genai {metadata.version('google-genai')} · {MODELO_GEMINI}"

    def preguntar(consulta, intentos=5):
        for intento in range(intentos):
            try:
                return cliente.models.generate_content(
                    model=MODELO_GEMINI, contents=consulta, config=types.GenerateContentConfig(temperature=0),
                ).text or ""
            except genai.errors.APIError as error:
                # 429: límite de consultas por minuto; 500 y 503: servidor ocupado. Otros errores (clave inválida) no se reintentan.
                if error.code not in (429, 500, 503) or intento == intentos - 1:
                    raise
                print(f"Gemini: error {error.code}; reintento en 30 s")
                time.sleep(30)

else:
    raise ValueError('LLM debe ser "ollama" o "gemini"')
print("Modelo de lenguaje:", VERSION_LLM)

## 3. Cargar el detector y el OCR

DocLayout-YOLO detecta las regiones. Solo se ordenan y transcriben las de texto corrido (títulos, textos y epígrafes); figuras, tablas y regiones `abandon` se guardan en la tabla, pero quedan fuera del orden de lectura.

In [ ]:
import numpy as np
import pytesseract
import torch
from doclayout_yolo import YOLOv10
from huggingface_hub import hf_hub_download

DISPOSITIVO = "cuda:0" if torch.cuda.is_available() else "cpu"
CONFIANZA = 0.20
RESOLUCION = 1024
CON_TEXTO = {"title", "plain text", "figure_caption", "table_caption", "table_footnote"}

detector = YOLOv10(hf_hub_download(repo_id="juliozhao/DocLayout-YOLO-DocStructBench", filename="doclayout_yolo_docstructbench_imgsz1024.pt"))

if OCR == "tesseract":
    VERSION_OCR = f"Tesseract {pytesseract.get_tesseract_version()} (spa)"

    def leer(recorte):
        return pytesseract.image_to_string(recorte, lang="spa", config="--psm 6").strip()

elif OCR == "paddle":
    import importlib.metadata as metadata

    import paddle
    from paddleocr import PaddleOCR

    ocr_paddle = PaddleOCR(
        lang="es", use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False,
        device="gpu:0" if paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0 else "cpu",
    )
    VERSION_OCR = f"PaddleOCR {metadata.version('paddleocr')} (PP-OCRv5, lang=es)"

    def leer(recorte):
        resultado = ocr_paddle.predict(np.array(recorte)[:, :, ::-1])[0]  # PaddleOCR espera colores BGR
        return "\n".join(resultado["rec_texts"]).strip()

else:
    raise ValueError('OCR debe ser "tesseract" o "paddle"')
print("Dispositivo:", DISPOSITIVO, "| OCR:", VERSION_OCR)

## 4. Pasos 1 y 2: detectar regiones y leer su texto

Se descartan las cajas contenidas en más de `SUPERPOSICION_MAXIMA` de su superficie dentro de otra con mayor confianza.

In [ ]:
SUPERPOSICION_MAXIMA = 0.8


def sin_superposiciones(cajas):
    """Descarta las cajas contenidas casi por completo en otra de mayor confianza."""
    conservadas = []
    for caja in sorted(cajas, key=lambda c: c["confianza"], reverse=True):
        area = (caja["x2"] - caja["x1"]) * (caja["y2"] - caja["y1"])
        if all(max(0, min(caja["x2"], otra["x2"]) - max(caja["x1"], otra["x1"])) * max(0, min(caja["y2"], otra["y2"]) - max(caja["y1"], otra["y1"]))
               < SUPERPOSICION_MAXIMA * area for otra in conservadas):
            conservadas.append(caja)
    return conservadas


def detectar_y_leer(pagina, imagen):
    """Regiones detectadas por DocLayout-YOLO; las que contienen texto corrido pasan por el OCR."""
    resultado = detector.predict(str(pagina), imgsz=RESOLUCION, conf=CONFIANZA, device=DISPOSITIVO, verbose=False)[0]
    cajas = []
    for xyxy, clase, confianza in zip(resultado.boxes.xyxy.cpu().numpy(), resultado.boxes.cls.cpu().numpy(), resultado.boxes.conf.cpu().numpy()):
        x1, y1, x2, y2 = (int(v) for v in xyxy)
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(imagen.width, x2), min(imagen.height, y2)
        if x2 > x1 and y2 > y1:
            cajas.append({"clase": detector.names[int(clase)], "confianza": round(float(confianza), 3), "x1": x1, "y1": y1, "x2": x2, "y2": y2})
    regiones = sin_superposiciones(cajas)
    for region in regiones:
        region["texto"] = leer(imagen.crop((region["x1"], region["y1"], region["x2"], region["y2"]))) if region["clase"] in CON_TEXTO else ""
    return regiones

## 5. Paso 3: agrupar en bloques

`AGRUPACION` elige cómo se agrupan las regiones:

- `"titulos"` (por defecto): cada región se asigna al **título más cercano que tiene encima en su misma columna**. Un título pegado a otro se une a su bloque. Las regiones sin título encima forman bloques por columna.
- `"espacios"`: corta la página en franjas horizontales separadas por espacios en blanco; las franjas bajas forman un solo bloque y las altas se dividen en columnas por los espacios verticales.

In [ ]:
AGRUPACION = "titulos"
TOLERANCIA = 10  # píxeles


def cruce_horizontal(a, b):
    return max(0, min(a["x2"], b["x2"]) - max(a["x1"], b["x1"]))


def columnas(regiones):
    """Agrupa regiones que se superponen horizontalmente (una columna) y las devuelve de izquierda a derecha."""
    grupos = []
    for region in sorted(regiones, key=lambda r: r["x1"]):
        destino = next((g for g in grupos if any(cruce_horizontal(region, otra) >= 0.5 * min(region["x2"] - region["x1"], otra["x2"] - otra["x1"]) for otra in g)), None)
        if destino is None:
            grupos.append([region])
        else:
            destino.append(region)
    return grupos


def agrupar_por_titulos(regiones):
    titulos = sorted((r for r in regiones if r["clase"] == "title"), key=lambda r: r["y1"])

    def titulo_encima(region, distancia=None):
        candidatos = [
            t for t in titulos
            if t is not region and t["y2"] <= region["y1"] + TOLERANCIA
            and cruce_horizontal(t, region) >= 0.3 * (region["x2"] - region["x1"])
            and (distancia is None or region["y1"] - t["y2"] <= distancia)
        ]
        return max(candidatos, key=lambda t: t["y2"], default=None)

    siguiente = 1
    for titulo in titulos:
        pegado = titulo_encima(titulo, distancia=titulo["y2"] - titulo["y1"])
        if pegado is not None:
            titulo["bloque"] = pegado["bloque"]
        else:
            titulo["bloque"], siguiente = siguiente, siguiente + 1
    huerfanas = []
    for region in regiones:
        if region["clase"] == "title":
            continue
        dueno = titulo_encima(region)
        if dueno is None:
            huerfanas.append(region)
        else:
            region["bloque"] = dueno["bloque"]
    for grupo in columnas(huerfanas):
        for region in grupo:
            region["bloque"] = siguiente
        siguiente += 1
    return regiones


def tramos(ocupado):
    """Tramos consecutivos de valores True en un vector: [(inicio, fin), ...]."""
    bordes = np.flatnonzero(np.diff(np.concatenate(([0], ocupado.astype(np.int8), [0]))))
    return list(zip(bordes[::2], bordes[1::2]))


def agrupar_por_espacios(regiones, ancho, alto):
    mapa = np.zeros((alto, ancho), dtype=bool)
    for region in regiones:
        mapa[region["y1"]:region["y2"], region["x1"]:region["x2"]] = True
    altura_tipica = np.median([r["y2"] - r["y1"] for r in regiones if r["clase"] != "title"] or [0])
    siguiente = 1
    for inicio, fin in tramos(mapa.any(axis=1)):
        en_franja = [r for r in regiones if inicio <= r["y1"] < fin]
        if fin - inicio < altura_tipica:
            partes = [en_franja]
        else:
            partes = [[r for r in en_franja if a <= (r["x1"] + r["x2"]) / 2 < b] for a, b in tramos(mapa[inicio:fin].any(axis=0))]
        for parte in partes:
            if parte:
                for region in parte:
                    region["bloque"] = siguiente
                siguiente += 1
    return regiones

## 6. Paso 4: ordenar dentro de cada bloque

Se corta el bloque por los espacios en blanco, alternando franjas horizontales y columnas, y se lee cada parte de arriba hacia abajo y de izquierda a derecha.

In [ ]:
def partir(regiones, eje):
    """Divide regiones por los huecos sobre un eje ("y": franjas; "x": columnas)."""
    inicio, fin = ("y1", "y2") if eje == "y" else ("x1", "x2")
    partes, limite = [], None
    for region in sorted(regiones, key=lambda r: r[inicio]):
        if limite is None or region[inicio] >= limite:
            partes.append([region])
            limite = region[fin]
        else:
            partes[-1].append(region)
            limite = max(limite, region[fin])
    return partes


def orden_local(regiones, eje="y"):
    if len(regiones) <= 1:
        return list(regiones)
    partes = partir(regiones, eje)
    if len(partes) == 1:
        eje = "x" if eje == "y" else "y"
        partes = partir(regiones, eje)
        if len(partes) == 1:
            return sorted(regiones, key=lambda r: (r["y1"], r["x1"]))
    siguiente_eje = "x" if eje == "y" else "y"
    return [r for parte in partes for r in orden_local(parte, siguiente_eje)]

## 7. Paso 5: ordenar los bloques con el modelo de lenguaje

Para cada par de bloques, el modelo recibe el **primer y el último párrafo** de cada uno y responde cuál se lee primero. Cada bloque suma un punto por comparación ganada y el orden final va de más a menos puntos; los empates los decide la posición en la página.

- **Orden sin LLM**: los bloques de arriba hacia abajo y de izquierda a derecha, para comparar.
- Con más de `MAX_BLOQUES_LLM` bloques, la página se ordena sin LLM (20 bloques son 190 consultas).
- En cada consulta, el bloque A es el que aparece primero en la página.

In [ ]:
import itertools

MAX_BLOQUES_LLM = 30
LARGO_FRAGMENTO = 500  # caracteres de cada párrafo que se envían al modelo

CONSULTA = """Eres un especialista en prensa histórica en español. Te muestro dos bloques de una misma página de diario. \
El texto proviene de un OCR y puede contener errores.

BLOQUE A
Primer párrafo: {a_inicio}
Último párrafo: {a_fin}

BLOQUE B
Primer párrafo: {b_inicio}
Último párrafo: {b_fin}

Según el sentido del texto, ¿cuál de los dos bloques se lee primero? Responde solo con la letra A o B."""


def extremos(regiones):
    textos = [r["texto"] for r in regiones if r["texto"]] or [""]
    return textos[0][:LARGO_FRAGMENTO], textos[-1][-LARGO_FRAGMENTO:]


def ordenar_bloques_con_llm(bloques, pagina):
    """bloques: lista de listas de regiones, en orden geométrico. Devuelve el nuevo orden y las decisiones."""
    puntos = [0] * len(bloques)
    decisiones = []
    for i, j in itertools.combinations(range(len(bloques)), 2):
        a_inicio, a_fin = extremos(bloques[i])
        b_inicio, b_fin = extremos(bloques[j])
        respuesta = preguntar(CONSULTA.format(a_inicio=a_inicio, a_fin=a_fin, b_inicio=b_inicio, b_fin=b_fin)).strip().upper()
        eleccion = next((letra for letra in respuesta if letra in "AB"), None)
        if eleccion == "A":
            puntos[i] += 1
        elif eleccion == "B":
            puntos[j] += 1
        decisiones.append({"pagina": pagina.name, "bloque_a": bloques[i][0]["bloque"], "bloque_b": bloques[j][0]["bloque"],
                           "respuesta": respuesta, "primero": {"A": bloques[i][0]["bloque"], "B": bloques[j][0]["bloque"]}.get(eleccion)})
    orden = sorted(range(len(bloques)), key=lambda k: (-puntos[k], k))
    return [bloques[k] for k in orden], decisiones

## 8. Procesar las páginas

En la imagen, cada color es un bloque y cada etiqueta indica `posición en la lectura · bloque`. La línea gris une las regiones en el orden final.

In [ ]:
import json

from PIL import ImageDraw, ImageFont

SALIDA = BASE / "resultados" / "06_orden_llm"
SALIDA.mkdir(parents=True, exist_ok=True)
COLORES = ["#e63946", "#1d3557", "#2a9d8f", "#e9c46a", "#8338ec", "#f4a261", "#06d6a0", "#ef476f", "#118ab2", "#bc6c25"]


def dibujar(imagen, ordenadas):
    vista = imagen.copy()
    lapiz = ImageDraw.Draw(vista)
    grosor = max(3, imagen.width // 500)
    fuente = ImageFont.load_default(size=max(16, imagen.width // 90))
    centros = [((r["x1"] + r["x2"]) / 2, (r["y1"] + r["y2"]) / 2) for r in ordenadas]
    if len(centros) > 1:
        lapiz.line(centros, fill="#555555", width=max(2, grosor // 2))
    for region in ordenadas:
        color = COLORES[(region["bloque"] - 1) % len(COLORES)]
        lapiz.rectangle((region["x1"], region["y1"], region["x2"], region["y2"]), outline=color, width=grosor)
        lapiz.text((region["x1"] + grosor, region["y1"] + grosor), f"{region['orden_con_llm']} · B{region['bloque']}",
                   fill=color, font=fuente, stroke_width=3, stroke_fill="white")
    return vista


filas, todas_las_decisiones = [], []
for pagina in IMAGENES:
    imagen = Image.open(pagina).convert("RGB")
    inicio = time.time()
    regiones = detectar_y_leer(pagina, imagen)
    con_texto = [r for r in regiones if r["clase"] in CON_TEXTO]
    if AGRUPACION == "titulos":
        agrupar_por_titulos(con_texto)
    elif AGRUPACION == "espacios":
        agrupar_por_espacios(con_texto, imagen.width, imagen.height)
    else:
        raise ValueError('AGRUPACION debe ser "titulos" o "espacios"')

    bloques = [orden_local([r for r in con_texto if r["bloque"] == b]) for b in sorted({r["bloque"] for r in con_texto})]
    bloques.sort(key=lambda bloque: (min(r["y1"] for r in bloque), min(r["x1"] for r in bloque)))
    sin_llm = [r for bloque in bloques for r in bloque]
    if 1 < len(bloques) <= MAX_BLOQUES_LLM:
        bloques_llm, decisiones = ordenar_bloques_con_llm(bloques, pagina)
    else:
        if len(bloques) > MAX_BLOQUES_LLM:
            print(f"{pagina.name}: {len(bloques)} bloques superan MAX_BLOQUES_LLM; se ordena sin LLM")
        bloques_llm, decisiones = bloques, []
    con_llm = [r for bloque in bloques_llm for r in bloque]
    for posicion, region in enumerate(sin_llm, 1):
        region["orden_sin_llm"] = posicion
    for posicion, region in enumerate(con_llm, 1):
        region["orden_con_llm"] = posicion
    todas_las_decisiones += decisiones

    destino = SALIDA / pagina.stem
    destino.mkdir(parents=True, exist_ok=True)
    vista = dibujar(imagen, con_llm)
    vista.save(destino / "orden_con_llm.png")
    transcripcion = []
    for bloque in bloques_llm:
        transcripcion.append(f"## Bloque {bloque[0]['bloque']}\n")
        transcripcion += [r["texto"] + "\n" for r in bloque if r["texto"]]
    (destino / "transcripcion.md").write_text("\n".join(transcripcion), encoding="utf-8")
    print(f"{pagina.name}: {len(regiones)} regiones, {len(con_texto)} con texto, {len(bloques)} bloques, "
          f"{len(decisiones)} consultas al LLM, {time.time() - inicio:.0f} s")
    mostrar(vista)

    for region in regiones:
        filas.append({"pagina": pagina.name, **region, "evaluacion_humana": ""})

COLUMNAS = ["pagina", "bloque", "orden_sin_llm", "orden_con_llm", "clase", "confianza", "x1", "y1", "x2", "y2", "texto", "evaluacion_humana"]
regiones = pd.DataFrame(filas, columns=COLUMNAS).astype({c: "Int64" for c in ("bloque", "orden_sin_llm", "orden_con_llm")})
regiones = regiones.sort_values(["pagina", "orden_con_llm"], na_position="last")
regiones.to_csv(SALIDA / "regiones.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
decisiones = pd.DataFrame(todas_las_decisiones, columns=["pagina", "bloque_a", "bloque_b", "respuesta", "primero"])
decisiones.to_csv(SALIDA / "decisiones_llm.csv", index=False, sep=";", encoding="utf-8-sig")
print("Guardado en:", SALIDA)

## 9. Comparar el orden sin y con modelo de lenguaje

La primera tabla muestra cada bloque con su posición en los dos órdenes y el comienzo de su texto. La segunda, cada decisión del modelo: qué par comparó, qué respondió y cuál eligió como primero. Si `respuesta` no contiene A ni B, la comparación no sumó puntos.

In [ ]:
pd.set_option("display.max_colwidth", 100)
con_bloque = regiones.dropna(subset=["bloque"])
if con_bloque.empty:
    print("No se detectaron regiones con texto con CONFIANZA =", CONFIANZA)
else:
    comparacion = con_bloque.groupby(["pagina", "bloque"]).agg(
        posicion_sin_llm=("orden_sin_llm", "min"),
        posicion_con_llm=("orden_con_llm", "min"),
        regiones=("clase", "count"),
        comienzo=("texto", lambda t: " ".join(t.dropna())[:100]),
    ).reset_index().sort_values(["pagina", "posicion_con_llm"])
    display(comparacion)
    display(decisiones)

## Registro de mediaciones

Guarda en `procesamiento.json` la herramienta, la versión, la configuración y el equipo con que se produjeron los resultados.

In [ ]:
import datetime
import importlib.metadata as metadata
import json
import platform

registro = {
    "herramienta": "Orden de lectura guiado por un LLM (reconstrucción de Vidal-Gorène et al., 2026)",
    "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
    "entorno": "Google Colab" if EN_COLAB else platform.platform(),
    "python": platform.python_version(),
    "dispositivo": DISPOSITIVO,
    "imagenes": [p.name for p in IMAGENES],
    "versiones": {p: metadata.version(p) for p in ("doclayout-yolo", "torch")},
    "modelos": {
        "regiones": "juliozhao/DocLayout-YOLO-DocStructBench (doclayout_yolo_docstructbench_imgsz1024.pt)",
        "ocr": VERSION_OCR,
        "llm": VERSION_LLM,
    },
    "parametros": {
        "confianza": CONFIANZA, "resolucion": RESOLUCION, "superposicion_maxima": SUPERPOSICION_MAXIMA, "agrupacion": AGRUPACION,
        "max_bloques_llm": MAX_BLOQUES_LLM, "largo_fragmento": LARGO_FRAGMENTO, "temperatura": 0,
    },
    "consulta": CONSULTA,
}
(SALIDA / "procesamiento.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(registro, ensure_ascii=False, indent=2))

## Descargar resultados

In [ ]:
import shutil

if EN_COLAB:
    from google.colab import files

    files.download(shutil.make_archive(f"/content/resultados_06_orden_llm", "zip", SALIDA))
else:
    print("Resultados en:", SALIDA.resolve())